In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pdal
import geopandas as gpd
import laspy
from laspy import CopcReader
import shapely
from shapely.geometry import box
from scipy.spatial import cKDTree
from concurrent.futures import ThreadPoolExecutor, as_completed

# Custom functions
from vineyard_analysis.config import DATA_DIR
from vineyard_analysis.io.aoc import load_aoc, load_spacing, filter_aoc
from vineyard_analysis.io.parcels import load_parcels, asign_aoc_to_parcels
from vineyard_analysis.io.zones import load_zones
from vineyard_analysis.lidar.download_all import download_all, merge_in_memory
from vineyard_analysis.analysis.row_analysis import find_row_orientation
from vineyard_analysis.analysis.process_parcel import process_parcel
from vineyard_analysis.analysis.clustering import cluster_points
from vineyard_analysis.lidar.lidar_file_urls import lidar_file_urls

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Load Variables
***
The following code chunk loads in all of the data needed for the analysis. Functions used for reading in data can be viewed in the python files located in ```src/vineyard_analysis/io```. Specifications for which data is being read in and how it is filtered can be changed in src ```src/vineyard_analysis/config.py```

In [3]:
aoc = filter_aoc(load_aoc().merge(load_spacing(), on="PDOid", how="left"))
zones = load_zones()
parcels = asign_aoc_to_parcels(load_parcels(), aoc)
parcels = parcels.sample(n=10000)

In [ ]:
output_csv = "parcel_results.csv"
results = []

# Keep this LOW — the IGN server rate-limits aggressively.
# Total in-flight requests = parcel workers × download workers (6 by default).
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {
        executor.submit(process_parcel, i, parcels, zones): i
        for i in range(len(parcels))
    }
    for future in as_completed(futures):
        idx = futures[future]
        try:
            res = future.result()
            results.append(res)
            print("\n".join(res["log"]))
        except Exception as e:
            print(f"Parcel {idx} failed: {e}")

results.sort(key=lambda r: (r["IDU"] is None, r["IDU"]))

fieldnames = [
    "IDU",
    "row_spacing",
    "plant_spacing",
    "rmse",
    "points_expected",
    "points_found",
    "area_missing_pct",
]

with open(output_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in results:
        writer.writerow({k: r[k] for k in fieldnames})

print(f"\nWrote {len(results)} rows to {output_csv}")